# CNU 학내정보 RAG 챗봇 Ver1 - 코랩 마스터 노트북

충남대학교 학생/교직원 12개 카테고리 질문에 환각 없이 답변하는 RAG 챗봇.

## 실행 방법
1. 런타임 → 런타임 유형 변경 → T4 GPU 선택
2. 셀 3에서 GitHub URL을 본인 레포로 교체
3. 런타임 → 모두 실행 (예상 소요 시간: 15~30분)

## 필수 환경변수
- `ANTHROPIC_API_KEY`: Claude API 키 (질문 생성, LLM judge 필요)
- `QUESTION_LIMIT`: 질문 풀 생성 수 (기본 500, 풀스케일 10000)
- `JUDGE_MOCK`: 1이면 API 없이 mock 점수 (테스트용)

## 1. 환경 셋업

In [ ]:
# USERNAME 을 본인 GitHub 계정으로 교체하세요
!git clone https://github.com/USERNAME/cnu-llm-bot.git
%cd cnu-llm-bot

In [ ]:
!pip install -q -r requirements.txt

## 2. 데이터 수집 (크롤링)

12개 카테고리 크롤러를 병렬 실행합니다. 사이트 오류 시 `_fallback()` 더미 데이터로 자동 폴백됩니다.

In [ ]:
from crawlers import run_all_crawlers
docs = run_all_crawlers()
print(f"수집된 문서: {len(docs)}")

## 3. LLM Distillation + Dedup (선택)

`DISTILL=1`로 설정하면 Claude Haiku로 핵심 내용을 추출합니다 (API 비용 발생).
기본값 `0`은 원문 그대로 사용합니다.

In [ ]:
import os
os.environ["DISTILL"] = "0"  # 비용 절약. 1로 바꾸면 Claude Haiku distillation 활성

from crawler_pipeline.distiller import distill_all
from crawler_pipeline.deduplicator import dedup

distilled = distill_all(docs) if os.environ.get("DISTILL") == "1" else docs
deduped = dedup(distilled)
print(f"중복 제거 후: {len(deduped)}")

## 4. 벡터 DB 구축

BGE-M3 임베딩 + Chroma persistent DB에 청크를 저장합니다.
모든 청크에 메타데이터 6종(source_url, data_category, last_crawled_at, valid_until, freshness_tier, original_text)이 강제 저장됩니다.

In [ ]:
from embedding.vector_store import build_vector_db, count
build_vector_db(deduped)
print(f"벡터 DB 청크 수: {count()}")

## 5. 1만 질문 풀 생성 (옵션) + 평가셋 200

`QUESTION_LIMIT=500`은 빠른 테스트용. `10000`으로 바꾸면 풀스케일 생성 (약 20~30분).
평가셋 200개는 스펙 분포로 추출: 시간성 50 / 학사 40 / 행정+장학 30 / 취업 25 / 학과 25 / 생활+시설 20 / 도메인 밖 10.

In [ ]:
import os
os.environ["QUESTION_LIMIT"] = "500"  # 풀스케일은 "10000"

from question_generation.question_pool_builder import generate_question_pool
from eval.testset_builder import build_testset

generate_question_pool(output_path="data/questions/question_pool.jsonl")
build_testset(
    question_pool_path="data/questions/question_pool.jsonl",
    output_path="data/testset.jsonl",
    n=200
)

## 6. 모델 로드 + GPU 메모리 측정

Qwen2.5-7B-Instruct-AWQ (4bit)를 로드합니다. OOM 시 자동으로 3B 모델로 폴백됩니다.
목표: 추론 메모리 < 12GB (T4 15GB 한도에 3GB 마진).

In [ ]:
from generation.llm import load_llm, print_gpu_memory
llm = load_llm()
print_gpu_memory()  # < 12GB 확인

## 7. 교수님 인터페이스 데모 (질문 파일 in → 답변 파일 out)

`answer_questions(questions_path, answers_path)` 함수가 교수님 제출 인터페이스입니다.
샘플 5개 질문으로 end-to-end 동작을 확인합니다.

In [ ]:
from interface.answer_questions import answer_questions
answer_questions("tests/sample_questions.jsonl", "data/sample_answers.jsonl")
!cat data/sample_answers.jsonl

## 8. 평가셋 200개 실행 + LLM-as-judge

3축 평가 (accuracy / relevance / domain_compliance, 각 0~5점).
`JUDGE_MOCK=1`로 설정하면 API 없이 더미 점수로 테스트 가능합니다.

In [ ]:
import os
os.environ["JUDGE_MOCK"] = "0"  # 1 로 바꾸면 mock 모드 (API 비용 없음)

from interface.answer_questions import answer_questions
from eval.evaluator import run_evaluation

answer_questions("data/testset.jsonl", "data/testset_answers.jsonl")
results = run_evaluation(
    testset_path="data/testset.jsonl",
    answers_path="data/testset_answers.jsonl"
)
print(results)

## 9. 추론 속도 벤치마크 (동점자 변별 대비)

50개 질문 기준 평균 응답 시간(avg_latency_ms)과 GPU 메모리를 측정합니다.

In [ ]:
from interface.benchmark import benchmark_inference
bench = benchmark_inference("data/testset.jsonl", n=50)
print(bench)

## 10. Gradio 데모 (시연용)

`share=True`로 실행되므로 `*.gradio.live` 공개 URL이 발급됩니다.
상위 5팀 발표 시연 및 데모에 사용합니다.

In [ ]:
from app.gradio_app import launch_app
launch_app()  # share=True → *.gradio.live URL 발급